### **TABL**

In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

from tqdm import tqdm 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
import torch
from torch.utils import data
import torch.nn as nn
import torch.optim as optim
import pickle
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torch.utils.data import Dataset as TorchDataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [22]:
class Dataset(TorchDataset):
    def __init__(self, x, y, num_classes, dim):

        self.num_classes = num_classes
        self.dim = dim

        x = torch.from_numpy(x).float()      
        y = torch.from_numpy(y).long()     

        self.x = x
        self.y = y

        self.length = x.shape[0] - dim + 1  


    def __len__(self):
        return self.length

    def __getitem__(self, i):

        w = self.x[i:i+self.dim, :]         

        w = w.permute(1, 0)                

        return w, self.y[i]

In [28]:

def make_loaders_for_fold(fold,h=5,dim=10,selected_dimension=144,batch_size=256,train_dir="../data/training",test_dir="../data/testing",val_ratio=0.0):
    train_path = f"{train_dir}/Train_Dst_NoAuction_ZScore_CF_{fold}.txt"
    test_path  = f"{test_dir}/Test_Dst_NoAuction_ZScore_CF_{fold}.txt"

    dec_data = np.loadtxt(train_path)
    dec_test = np.loadtxt(test_path)

    if val_ratio is None or val_ratio <= 0.0:
        dec_train_raw = dec_data
        dec_val_raw = None
    else:
        n_cols = dec_data.shape[1]
        cut = int(n_cols * (1 - val_ratio))
        dec_train_raw = dec_data[:, :cut]
        dec_val_raw   = dec_data[:, cut:]

    y_train = dec_train_raw[-h, :].flatten()[dim-1:] - 1
    y_test  = dec_test[-h, :].flatten()[dim-1:] - 1

    X_train = dec_train_raw[:selected_dimension, :].T
    X_test  = dec_test[:selected_dimension, :].T

    if X_train.shape[0] < dim:
        raise ValueError(f"Train too short: X_train has N={X_train.shape[0]} but dim={dim}")
    if X_test.shape[0] < dim:
        raise ValueError(f"Test too short: X_test has N={X_test.shape[0]} but dim={dim}")

    ds_train = Dataset(X_train, y_train, 3, dim)
    ds_test  = Dataset(X_test,  y_test,  3, dim)

    train_loader = DataLoader(ds_train, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False)

    if dec_val_raw is None:
        val_loader = None
    else:
        y_val = dec_val_raw[-h, :].flatten()[dim-1:] - 1
        X_val = dec_val_raw[:selected_dimension, :].T

        if X_val.shape[0] < dim:
            val_loader = None
        else:
            ds_val = Dataset(X_val, y_val, 3, dim)
            val_loader = DataLoader(ds_val, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, y_train

In [5]:
def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [6]:
def compute_class_weights(y_np):

    c0 = (y_np == 0).sum()
    c1 = (y_np == 1).sum()
    c2 = (y_np == 2).sum()
    w = torch.tensor([1e6/max(c0,1), 1e6/max(c1,1), 1e6/max(c2,1)], dtype=torch.float32, device=device)
    return w

In [24]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for X, y in loader:
        X = X.float().to(device)
        y = y.long().to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        preds = torch.argmax(logits, dim=1)
        acc = (preds == y).float().mean().item()

        total_loss += loss.item()
        total_acc += acc
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

In [30]:

def accuracy_from_logits(logits, y):
    preds = torch.argmax(logits, dim=1)
    return (preds == y).float().mean().item()

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_true = []
    n_batches = 0

    for X, y in loader:
        X = X.float().to(device)
        y = y.long().to(device)

        logits = model(X)
        loss = criterion(logits, y)

        preds = torch.argmax(logits, dim=1)

        total_loss += loss.item()
        n_batches += 1

        all_preds.append(preds.detach().cpu().numpy())
        all_true.append(y.detach().cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_true  = np.concatenate(all_true)

    acc = (all_preds == all_true).mean()
    macro_f1 = f1_score(all_true, all_preds, average="macro")

    return total_loss / n_batches, acc, macro_f1


In [9]:
class MLPBaseline(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),              
            nn.Linear(144*10, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [10]:
class CNNTimeBaseline(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels=144, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        
        z = self.features(x)    
        z = z.mean(dim=2)             
        out = self.classifier(z)      
        return out

In [11]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for X, y in loader:
        X = X.float().to(device)
        y = y.long().to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [12]:
def fit(model, train_loader, val_loader, criterion, optimizer, epochs=20):
    best_val_f1 = -1.0
    best_state = None

    for epoch in range(1, epochs+1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, va_f1 = eval_model(model, val_loader, criterion)

        print(f"Epoch {epoch:03d} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_acc={va_acc:.4f} | val_macroF1={va_f1:.4f}")

        if va_f1 > best_val_f1:
            best_val_f1 = va_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

In [ ]:
def run_walk_forward_cv(model_builder, model_name, epochs=20, lr=1e-3, batch_size=256, h=5, dim=10, val_ratio=0.0):
 
    set_seed(42)
    fold_results = []

    print(f"\n==============================")
    print(f"MODEL (Setup1 walk-forward): {model_name}")
    print(f"==============================\n")

    for fold in range(1, 10):
        print(f"\n===== Fold {fold}: Train days 1..{fold} | Test day {fold+1} =====")

        train_loader, val_loader, test_loader, y_train = make_loaders_for_fold(
            fold=fold, h=h, dim=dim, batch_size=batch_size, val_ratio=val_ratio
        )

        weights = compute_class_weights(y_train.astype(np.int64))

        weights = weights / weights.mean()
        criterion = nn.CrossEntropyLoss(weight=weights)

        model = model_builder().to(device)
        optimizer = optim.Adam(model.parameters(), lr=lr)

        best_state = None
        best_val_f1 = -1.0

        for epoch in range(1, epochs+1):
            tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)

            if val_loader is not None:
                va_loss, va_acc, va_f1 = eval_epoch(model, val_loader, criterion)
                print(f"Epoch {epoch:03d} | tr_loss {tr_loss:.4f} acc {tr_acc:.4f} | val_loss {va_loss:.4f} acc {va_acc:.4f} macroF1 {va_f1:.4f}")
                if va_f1 > best_val_f1:
                    best_val_f1 = va_f1
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                print(f"Epoch {epoch:03d} | tr_loss {tr_loss:.4f} acc {tr_acc:.4f}")

        if best_state is not None:
            model.load_state_dict(best_state)

        te_loss, te_acc, te_f1 = eval_epoch(model, test_loader, criterion)
        print(f"Fold {fold} TEST | loss {te_loss:.4f} acc {te_acc:.4f} macroF1 {te_f1:.4f}")

        fold_results.append({"fold": fold, "test_loss": te_loss, "test_acc": te_acc, "test_macroF1": te_f1})

    accs = [r["test_acc"] for r in fold_results]
    f1s  = [r["test_macroF1"] for r in fold_results]

    print(f"\n===== WALK-FORWARD SUMMARY: {model_name} =====")
    print(f"Test Acc    mean={np.mean(accs):.4f} std={np.std(accs, ddof=0):.4f}")
    print(f"Test MacroF1 mean={np.mean(f1s):.4f} std={np.std(f1s, ddof=0):.4f}")

    return fold_results


In [14]:
class LSTMClassifier(nn.Module):
    def __init__(self, num_classes=3, hidden_size=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=144,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.0 if num_layers == 1 else dropout
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, (h_n, c_n) = self.lstm(x)
        h_last = h_n[-1] 
        return self.head(h_last)

In [15]:
class TCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.res = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.net(x) + self.res(x)

class TCNClassifier(nn.Module):
    def __init__(self, num_classes=3, dropout=0.2):
        super().__init__()
        self.tcn = nn.Sequential(
            TCNBlock(144, 128, kernel_size=3, dilation=1, dropout=dropout),
            TCNBlock(128, 128, kernel_size=3, dilation=2, dropout=dropout),
            TCNBlock(128, 128, kernel_size=3, dilation=4, dropout=dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        z = self.tcn(x)   
        z = z.mean(dim=2)      
        return self.head(z)

In [36]:
def summarize_cv(fold_results, model_name):
    df = pd.DataFrame(fold_results)
    summary = {
        "model": model_name,
        "acc_mean": df["test_acc"].mean(),
        "acc_std": df["test_acc"].std(ddof=0),
        "f1_mean": df["test_macroF1"].mean(),
        "f1_std": df["test_macroF1"].std(ddof=0),
    }
    return df, summary

In [31]:
mlp_results = run_walk_forward_cv(
    model_builder=lambda: MLPBaseline(num_classes=3),
    model_name="MLP Baseline",
    epochs=20,
    lr=1e-3,
    val_ratio=0.0 
)


MODEL (Setup1 walk-forward): MLP Baseline


===== Fold 1: Train days 1..1 | Test day 2 =====
Epoch 001 | tr_loss 0.9990 acc 0.5195
Epoch 002 | tr_loss 0.9025 acc 0.5949
Epoch 003 | tr_loss 0.8543 acc 0.6192
Epoch 004 | tr_loss 0.8124 acc 0.6419
Epoch 005 | tr_loss 0.7762 acc 0.6624
Epoch 006 | tr_loss 0.7464 acc 0.6741
Epoch 007 | tr_loss 0.7182 acc 0.6884
Epoch 008 | tr_loss 0.6899 acc 0.6983
Epoch 009 | tr_loss 0.6722 acc 0.7082
Epoch 010 | tr_loss 0.6395 acc 0.7196
Epoch 011 | tr_loss 0.6136 acc 0.7281
Epoch 012 | tr_loss 0.6011 acc 0.7324
Epoch 013 | tr_loss 0.5760 acc 0.7425
Epoch 014 | tr_loss 0.5533 acc 0.7512
Epoch 015 | tr_loss 0.5412 acc 0.7567
Epoch 016 | tr_loss 0.5129 acc 0.7663
Epoch 017 | tr_loss 0.5097 acc 0.7688
Epoch 018 | tr_loss 0.4829 acc 0.7809
Epoch 019 | tr_loss 0.4796 acc 0.7799
Epoch 020 | tr_loss 0.4565 acc 0.7915
Fold 1 TEST | loss 1.1451 acc 0.6192 macroF1 0.5773

===== Fold 2: Train days 1..2 | Test day 3 =====
Epoch 001 | tr_loss 0.9740 acc 0.5402
Epoch 

In [32]:
cnn_results = run_walk_forward_cv(
    model_builder=lambda: CNNTimeBaseline(num_classes=3),
    model_name="CNN Time Baseline",
    epochs=20,
    lr=1e-3
)


MODEL (Setup1 walk-forward): CNN Time Baseline


===== Fold 1: Train days 1..1 | Test day 2 =====
Epoch 001 | tr_loss 1.0297 acc 0.4949
Epoch 002 | tr_loss 0.9685 acc 0.5391
Epoch 003 | tr_loss 0.9067 acc 0.5985
Epoch 004 | tr_loss 0.8425 acc 0.6374
Epoch 005 | tr_loss 0.7841 acc 0.6689
Epoch 006 | tr_loss 0.7333 acc 0.6926
Epoch 007 | tr_loss 0.7006 acc 0.7088
Epoch 008 | tr_loss 0.6748 acc 0.7214
Epoch 009 | tr_loss 0.6478 acc 0.7323
Epoch 010 | tr_loss 0.6226 acc 0.7449
Epoch 011 | tr_loss 0.5947 acc 0.7563
Epoch 012 | tr_loss 0.5705 acc 0.7653
Epoch 013 | tr_loss 0.5534 acc 0.7711
Epoch 014 | tr_loss 0.5257 acc 0.7851
Epoch 015 | tr_loss 0.5092 acc 0.7913
Epoch 016 | tr_loss 0.4850 acc 0.8001
Epoch 017 | tr_loss 0.4614 acc 0.8090
Epoch 018 | tr_loss 0.4459 acc 0.8147
Epoch 019 | tr_loss 0.4232 acc 0.8216
Epoch 020 | tr_loss 0.4097 acc 0.8274
Fold 1 TEST | loss 0.9253 acc 0.6634 macroF1 0.6283

===== Fold 2: Train days 1..2 | Test day 3 =====
Epoch 001 | tr_loss 1.0172 acc 0.5086
E

In [33]:
lstm_results = run_walk_forward_cv(
    model_builder=lambda: LSTMClassifier(num_classes=3, hidden_size=128),
    model_name="LSTM",
    epochs=20,
    lr=1e-3
)


MODEL (Setup1 walk-forward): LSTM


===== Fold 1: Train days 1..1 | Test day 2 =====
Epoch 001 | tr_loss 0.9902 acc 0.5213
Epoch 002 | tr_loss 0.8317 acc 0.6376
Epoch 003 | tr_loss 0.7468 acc 0.6831
Epoch 004 | tr_loss 0.6810 acc 0.7147
Epoch 005 | tr_loss 0.6266 acc 0.7340
Epoch 006 | tr_loss 0.5726 acc 0.7539
Epoch 007 | tr_loss 0.5221 acc 0.7752
Epoch 008 | tr_loss 0.4736 acc 0.7935
Epoch 009 | tr_loss 0.4344 acc 0.8099
Epoch 010 | tr_loss 0.3876 acc 0.8292
Epoch 011 | tr_loss 0.3491 acc 0.8429
Epoch 012 | tr_loss 0.3191 acc 0.8562
Epoch 013 | tr_loss 0.2847 acc 0.8700
Epoch 014 | tr_loss 0.2481 acc 0.8857
Epoch 015 | tr_loss 0.2238 acc 0.8956
Epoch 016 | tr_loss 0.1933 acc 0.9080
Epoch 017 | tr_loss 0.1832 acc 0.9160
Epoch 018 | tr_loss 0.1644 acc 0.9215
Epoch 019 | tr_loss 0.1457 acc 0.9289
Epoch 020 | tr_loss 0.1278 acc 0.9390
Fold 1 TEST | loss 2.1828 acc 0.6583 macroF1 0.6032

===== Fold 2: Train days 1..2 | Test day 3 =====
Epoch 001 | tr_loss 0.9208 acc 0.5872
Epoch 002 | tr

In [34]:
tcn_results = run_walk_forward_cv(
    model_builder=lambda: TCNClassifier(num_classes=3),
    model_name="TCN",
    epochs=20,
    lr=1e-3
)



MODEL (Setup1 walk-forward): TCN


===== Fold 1: Train days 1..1 | Test day 2 =====
Epoch 001 | tr_loss 1.0048 acc 0.5159
Epoch 002 | tr_loss 0.8986 acc 0.6028
Epoch 003 | tr_loss 0.8235 acc 0.6425
Epoch 004 | tr_loss 0.7762 acc 0.6663
Epoch 005 | tr_loss 0.7429 acc 0.6844
Epoch 006 | tr_loss 0.7166 acc 0.6955
Epoch 007 | tr_loss 0.6852 acc 0.7088
Epoch 008 | tr_loss 0.6627 acc 0.7208
Epoch 009 | tr_loss 0.6333 acc 0.7340
Epoch 010 | tr_loss 0.6062 acc 0.7450
Epoch 011 | tr_loss 0.5919 acc 0.7467
Epoch 012 | tr_loss 0.5687 acc 0.7594
Epoch 013 | tr_loss 0.5464 acc 0.7635
Epoch 014 | tr_loss 0.5302 acc 0.7699
Epoch 015 | tr_loss 0.5090 acc 0.7779
Epoch 016 | tr_loss 0.5055 acc 0.7802
Epoch 017 | tr_loss 0.4871 acc 0.7857
Epoch 018 | tr_loss 0.4671 acc 0.7960
Epoch 019 | tr_loss 0.4545 acc 0.8014
Epoch 020 | tr_loss 0.4434 acc 0.8030
Fold 1 TEST | loss 0.9030 acc 0.7038 macroF1 0.6614

===== Fold 2: Train days 1..2 | Test day 3 =====
Epoch 001 | tr_loss 0.9678 acc 0.5553
Epoch 002 | tr_

In [37]:
df_mlp, sum_mlp = summarize_cv(mlp_results, "MLP")
df_cnn, sum_cnn = summarize_cv(cnn_results, "CNN")
df_lstm, sum_lstm = summarize_cv(lstm_results, "LSTM")
df_tcn, sum_tcn = summarize_cv(tcn_results, "TCN")

In [ ]:
summary_df = pd.DataFrame([sum_mlp, sum_cnn, sum_lstm, sum_tcn]).sort_values("f1_mean", ascending=False)
summary_df

,model,acc_mean,acc_std,f1_mean,f1_std
3,TCN,0.800810,0.044219,0.755496,0.038201
1,CNN,0.789705,0.052659,0.743444,0.043868
0,MLP,0.753220,0.065372,0.698131,0.055158
2,LSTM,0.749135,0.053985,0.695613,0.047659
